# N-Lens System Inversion

Fit lens column parameters using the Glaser bell model:
- **Focal length**: $1/f_i = C_{f,i} \cdot (I_{0,i}(1+w))^2$
- **Rotation**: $\psi_i = K_v \cdot I_{0,i}(1+w)$

Using known rotation constant $K_v$ (from 200 kV accelerating voltage) to break degeneracies.

In [42]:
import sys
sys.path.insert(0, '../../src')

import jax
import jax.numpy as jnp
import numpy as np
from scipy.optimize import least_squares as scipy_least_squares
import time

from temgym_core.transfer_matrices import propagation_matrix, lens_matrix

jax.config.update("jax_enable_x64", True)
print("Imports successful")

Imports successful


In [43]:
# Physical constants
E_CHARGE = 1.602176634e-19
M_E = 9.1093837015e-31
C_LIGHT = 299792458.0
MU_0 = 1.25663706212e-6

U_ACCEL = 200e3  # 200 kV
gamma_rel = 1 + E_CHARGE * U_ACCEL / (M_E * C_LIGHT**2)
v_electron = C_LIGHT * np.sqrt(1 - 1 / gamma_rel**2)
K_ROT = E_CHARGE * MU_0 / (2 * M_E * v_electron)  # rad/AT

print(f"K_rot = {K_ROT:.6e} rad/AT")

K_rot = 5.301506e-04 rad/AT


In [44]:
# Forward model
def f_from_excitation(Cf_i, I0_i, w):
    return 1.0 / (Cf_i * (I0_i * (1.0 + w)) ** 2)

def psi_from_excitation(Kv, I0_i, w):
    return Kv * I0_i * (1.0 + w)

def build_abcd(dists, focals, xp=jnp):
    M = propagation_matrix(dists[-1], xp=xp)
    for i in reversed(range(len(focals))):
        M = M @ lens_matrix(focals[i], xp=xp)
        M = M @ propagation_matrix(dists[i], xp=xp)
    return M

def compute_AB(dists, focals):
    M = build_abcd(dists, focals, xp=jnp)
    return M[0, 0], M[0, 1]

def model_measurement(dists, I0, Cf, Kv, wobble_lens, w, defocus):
    f_use = []
    for i in range(len(I0)):
        w_i = jnp.where(i == wobble_lens, w, 0.0)
        f_use.append(f_from_excitation(Cf[i], I0[i], w_i))
    f_use = jnp.array(f_use)

    d_use = jnp.array(dists)
    d_use = d_use.at[0].add(defocus)

    A, B = compute_AB(d_use, f_use)

    psi_total = 0.0
    for i in range(len(I0)):
        w_i = jnp.where(i == wobble_lens, w, 0.0)
        psi_total = psi_total + psi_from_excitation(Kv, I0[i], w_i)

    return A, B, psi_total

def generate_measurements(d_true, I0_true, Cf_true, Kv, wobble_values, defocus_values):
    wl_list, w_list, df_list = [], [], []
    A_list, B_list, psi_list = [], [], []

    for lens_idx in range(len(I0_true)):
        for w in wobble_values:
            for defocus in defocus_values:
                A, B, psi = model_measurement(
                    d_true, I0_true, Cf_true, Kv, lens_idx, w, defocus
                )
                wl_list.append(lens_idx)
                w_list.append(float(w))
                df_list.append(float(defocus))
                A_list.append(float(A))
                B_list.append(float(B))
                psi_list.append(float(psi))

    return {
        "wobble_lens": jnp.array(wl_list, dtype=jnp.int32),
        "wobble": jnp.array(w_list),
        "defocus": jnp.array(df_list),
        "A": jnp.array(A_list),
        "B": jnp.array(B_list),
        "psi": jnp.array(psi_list),
    }

print("Forward model ready")

Forward model ready


In [45]:
# Residual functions
def make_residual_fn(measurements, n_lenses, scales, Kv):
    """vmap-vectorized residual function (Glaser parametrisation).

    Parameter vector: [d1,...,d_{N+1}, I0_1,...,I0_N, Cf_1,...,Cf_N]
    Total: 3N+1 parameters.

    K_v is KNOWN (not fitted) — eliminates the f ↔ ψ degeneracy.
    """
    meas_wl  = measurements["wobble_lens"]
    meas_w   = measurements["wobble"]
    meas_df  = measurements["defocus"]
    meas_A   = measurements["A"]
    meas_B   = measurements["B"]
    meas_psi = measurements["psi"]

    A_scale, B_scale, psi_scale = scales
    n_dist = n_lenses + 1
    n_I0   = n_lenses

    @jax.jit
    def residual_fn(params):
        d  = params[:n_dist]
        I0 = params[n_dist : n_dist + n_I0]
        Cf = params[n_dist + n_I0 : n_dist + 2 * n_I0]

        def single_pred(wl, w, df):
            return model_measurement(d, I0, Cf, Kv, wl, w, df)

        A_pred, B_pred, psi_pred = jax.vmap(single_pred)(meas_wl, meas_w, meas_df)

        res_A   = (A_pred   - meas_A)   / A_scale
        res_B   = (B_pred   - meas_B)   / B_scale
        res_psi = (psi_pred - meas_psi) / psi_scale

        return jnp.concatenate([res_A, res_B, res_psi])

    return residual_fn

def make_residual_fn_constrained(measurements, n_lenses, scales, Kv, total_distance=None):
    """vmap-vectorized residual function with optional total distance constraint.

    If total_distance is specified:
        Parameter vector: [d1,...,d_N, I0_1,...,I0_N, Cf_1,...,Cf_N]
        where d_{N+1} = total_distance - sum(d_1...d_N)
        Total: 3N parameters (one less distance)
    
    If total_distance is None:
        Parameter vector: [d1,...,d_{N+1}, I0_1,...,I0_N, Cf_1,...,Cf_N]
        Total: 3N+1 parameters (standard)

    K_v is KNOWN (not fitted) — eliminates the f ↔ ψ degeneracy.
    """
    meas_wl  = measurements["wobble_lens"]
    meas_w   = measurements["wobble"]
    meas_df  = measurements["defocus"]
    meas_A   = measurements["A"]
    meas_B   = measurements["B"]
    meas_psi = measurements["psi"]

    A_scale, B_scale, psi_scale = scales
    n_I0   = n_lenses
    
    if total_distance is not None:
        n_dist_fit = n_lenses  # Only fit N distances, derive the (N+1)th
    else:
        n_dist_fit = n_lenses + 1  # Fit all N+1 distances

    @jax.jit
    def residual_fn(params):
        if total_distance is not None:
            # Reconstruct full distance array from first N + constraint
            d_fit = params[:n_dist_fit]
            d_last = total_distance - jnp.sum(d_fit)
            d = jnp.concatenate([d_fit, jnp.array([d_last])])
        else:
            d = params[:n_dist_fit]
        
        I0 = params[n_dist_fit : n_dist_fit + n_I0]
        Cf = params[n_dist_fit + n_I0 : n_dist_fit + 2 * n_I0]

        def single_pred(wl, w, df):
            return model_measurement(d, I0, Cf, Kv, wl, w, df)

        A_pred, B_pred, psi_pred = jax.vmap(single_pred)(meas_wl, meas_w, meas_df)

        res_A   = (A_pred   - meas_A)   / A_scale
        res_B   = (B_pred   - meas_B)   / B_scale
        res_psi = (psi_pred - meas_psi) / psi_scale

        return jnp.concatenate([res_A, res_B, res_psi])

    return residual_fn

def make_residual_fn_no_B(measurements, n_lenses, scales, Kv):
    """vmap-vectorized residual function WITHOUT B terms (only A and ψ).

    Parameter vector: [d1,...,d_{N+1}, I0_1,...,I0_N, Cf_1,...,Cf_N]
    Total: 3N+1 parameters.

    K_v is KNOWN (not fitted) — eliminates the f ↔ ψ degeneracy.
    """
    meas_wl  = measurements["wobble_lens"]
    meas_w   = measurements["wobble"]
    meas_df  = measurements["defocus"]
    meas_A   = measurements["A"]
    meas_psi = measurements["psi"]

    A_scale, psi_scale = scales  # Only two scales now
    n_dist = n_lenses + 1
    n_I0   = n_lenses

    @jax.jit
    def residual_fn(params):
        d  = params[:n_dist]
        I0 = params[n_dist : n_dist + n_I0]
        Cf = params[n_dist + n_I0 : n_dist + 2 * n_I0]

        def single_pred(wl, w, df):
            A_pred, B_pred, psi_pred = model_measurement(d, I0, Cf, Kv, wl, w, df)
            return A_pred, psi_pred  # Only return A and psi

        A_pred, psi_pred = jax.vmap(single_pred)(meas_wl, meas_w, meas_df)

        res_A   = (A_pred - meas_A) / A_scale
        res_psi = (psi_pred - meas_psi) / psi_scale

        return jnp.concatenate([res_A, res_psi])

    return residual_fn

def make_residual_fn_constrained_no_B(measurements, n_lenses, scales, Kv, total_distance=None):
    """vmap-vectorized residual function with total distance constraint, NO B terms.

    If total_distance is specified:
        Parameter vector: [d1,...,d_N, I0_1,...,I0_N, Cf_1,...,Cf_N]
        where d_{N+1} = total_distance - sum(d_1...d_N)
        Total: 3N parameters (one less distance)
    
    If total_distance is None:
        Parameter vector: [d1,...,d_{N+1}, I0_1,...,I0_N, Cf_1,...,Cf_N]
        Total: 3N+1 parameters (standard)

    K_v is KNOWN (not fitted) — eliminates the f ↔ ψ degeneracy.
    """
    meas_wl  = measurements["wobble_lens"]
    meas_w   = measurements["wobble"]
    meas_df  = measurements["defocus"]
    meas_A   = measurements["A"]
    meas_psi = measurements["psi"]

    A_scale, psi_scale = scales  # Only two scales now
    n_I0   = n_lenses
    
    if total_distance is not None:
        n_dist_fit = n_lenses  # Only fit N distances, derive the (N+1)th
    else:
        n_dist_fit = n_lenses + 1  # Fit all N+1 distances

    @jax.jit
    def residual_fn(params):
        if total_distance is not None:
            # Reconstruct full distance array from first N + constraint
            d_fit = params[:n_dist_fit]
            d_last = total_distance - jnp.sum(d_fit)
            d = jnp.concatenate([d_fit, jnp.array([d_last])])
        else:
            d = params[:n_dist_fit]
        
        I0 = params[n_dist_fit : n_dist_fit + n_I0]
        Cf = params[n_dist_fit + n_I0 : n_dist_fit + 2 * n_I0]

        def single_pred(wl, w, df):
            A_pred, B_pred, psi_pred = model_measurement(d, I0, Cf, Kv, wl, w, df)
            return A_pred, psi_pred  # Only return A and psi

        A_pred, psi_pred = jax.vmap(single_pred)(meas_wl, meas_w, meas_df)

        res_A   = (A_pred - meas_A) / A_scale
        res_psi = (psi_pred - meas_psi) / psi_scale

        return jnp.concatenate([res_A, res_psi])

    return residual_fn

def add_measurement_noise(measurements, A_noise_frac=0.0, psi_noise_frac=0.0, distance_noise_frac=0.0, rng_seed=None):
    """Add realistic noise to synthetic measurements.
    
    Parameters:
    -----------
    measurements : dict
        Output from generate_measurements()
    A_noise_frac : float
        Fractional noise level for A (e.g., 0.01 = 1%)
    psi_noise_frac : float
        Fractional noise level for ψ (e.g., 0.01 = 1%)
    distance_noise_frac : float
        Fractional noise level for total_distance (e.g., 0.01 = 1%)
    rng_seed : int, optional
        Random seed for reproducibility
    
    Returns:
    --------
    measurements_noisy : dict
        Noisy copy of measurements
    """
    if rng_seed is not None:
        rng = np.random.default_rng(rng_seed)
    else:
        rng = np.random.default_rng()
    
    measurements_noisy = measurements.copy()
    
    # Add noise to A values
    if A_noise_frac > 0:
        A_noise = rng.normal(0, A_noise_frac * np.abs(measurements['A']))
        measurements_noisy['A'] = measurements['A'] + A_noise
    
    # Add noise to ψ values
    if psi_noise_frac > 0:
        psi_noise = rng.normal(0, psi_noise_frac * np.abs(measurements['psi']))
        measurements_noisy['psi'] = measurements['psi'] + psi_noise
    
    # Note: distance_noise affects the constraint during fitting
    # Store it separately for use in fit_system_no_B
    measurements_noisy['distance_noise_frac'] = distance_noise_frac
    measurements_noisy['distance_noise_seed'] = rng_seed if rng_seed is not None else 0
    
    return measurements_noisy

print("Residual functions ready")

Residual functions ready


## Quick Start: Test Different Configurations

Edit the three configuration lines below, then run the cell.

**Note:** For best results:
- **N=2 or N=3** work reliably
- **N=4, N=5** may require additional refinement (see OLD notebook for advanced fit_system)
- **Distance constraint is recommended** (enables underdetermined systems to converge)
- **with_B approach is more stable** than without_B (more measurement information)

In [ ]:
# ================================================================
# CONFIGURATION: Edit these
# ================================================================

N_LENSES = 3                    # 2, 3, 4, or 5 (2-3 are most reliable)
MEASUREMENT_TYPE = 'clean'        # 'clean' or 'noisy'
FITTING_APPROACH = 'without_B'       # 'with_B' or 'without_B' (with_B is more stable)

# Noise parameters (only if MEASUREMENT_TYPE='noisy')
A_NOISE = 0.05                    # 5% noise on A
PSI_NOISE = 0.05                  # 5% noise on psi
NOISE_SEED = 42

# Number of optimization starts (small for quick test)
N_STARTS = 40

# Refinement settings (additional optimization passes if fit is poor)
REFINE_RUNS = 20                   # Number of refinement passes

# ================================================================
# SYSTEM DEFINITIONS
# ================================================================

SYSTEMS = {
    2: {
        'D': np.array([2e-3, 40e-3, 200e-3]),
        'I0': np.array([2000.0, 800.0]),
        'Cf': np.array([8.333e-5, 3.125e-5]),
    },
    3: {
        'D': np.array([2e-3, 20e-3, 40e-3, 200e-3]),
        'I0': np.array([2000.0, 1000.0, 600.0]),
        'Cf': np.array([8.333e-5, 5.0e-5, 3.472e-5]),
    },
    4: {
        'D': np.array([2e-3, 10e-3, 20e-3, 40e-3, 200e-3]),
        'I0': np.array([2000.0, 1200.0, 800.0, 500.0]),
        'Cf': np.array([8.333e-5, 5.787e-5, 3.906e-5, 2.441e-5]),
    },
    5: {
        'D': np.array([2e-3, 8e-3, 15e-3, 25e-3, 40e-3, 200e-3]),
        'I0': np.array([3400.0, 3200.0, 3000.0, 2800.0, 2600.0]),
        'Cf': np.array([8.333e-5, 5.415e-5, 3.951e-5, 2.755e-5, 2.0e-5]),
    },
}

# ================================================================
# RUN FITTING
# ================================================================

wobble_vals = np.linspace(-0.1, 0.1, 25)
defocus_vals = np.linspace(0, 0.0, 1)

sys_params = SYSTEMS[N_LENSES]
d_true = sys_params['D']
i0_true = sys_params['I0']
cf_true = sys_params['Cf']

print("="*70)
print(f"N={N_LENSES} Lenses | {MEASUREMENT_TYPE.upper()} | {FITTING_APPROACH.upper()}")
print("="*70)

# Generate measurements
meas_clean = generate_measurements(d_true, i0_true, cf_true, K_ROT, wobble_vals, defocus_vals)
if MEASUREMENT_TYPE == 'noisy':
    meas = add_measurement_noise(meas_clean, A_noise_frac=A_NOISE, psi_noise_frac=PSI_NOISE, rng_seed=NOISE_SEED)
    print(f"\nNoise: {A_NOISE*100:.1f}% A + {PSI_NOISE*100:.1f}% psi")
else:
    meas = meas_clean
    print(f"\nMeasurements: Clean")

# Create residual function
if FITTING_APPROACH == 'without_B':
    scales = (max(np.max(np.abs(meas["A"])), 1e-12), max(np.max(np.abs(meas["psi"])), 1e-12))
    rfn = make_residual_fn_constrained_no_B(meas, N_LENSES, scales, K_ROT, total_distance=d_true.sum())
    print(f"Residuals: A + ψ ({3*N_LENSES} params, with distance constraint)")
else:
    scales = (max(np.max(np.abs(meas["A"])), 1e-12), max(np.max(np.abs(meas["B"])), 1e-12), max(np.max(np.abs(meas["psi"])), 1e-12))
    rfn = make_residual_fn_constrained(meas, N_LENSES, scales, K_ROT, total_distance=d_true.sum())
    print(f"Residuals: A + B + ψ ({3*N_LENSES} params, with distance constraint)")

# Optimization bounds
lower = np.concatenate([np.full(N_LENSES, 1e-3), np.full(N_LENSES, 10.0), np.full(N_LENSES, 1e-6)])
upper = np.concatenate([np.full(N_LENSES, 0.5), np.full(N_LENSES, 30000.0), np.full(N_LENSES, 1e-4)])

# Initial guess
rng = np.random.default_rng(123)
d_init = d_true.sum() * (rng.random(N_LENSES) + 0.1)
d_init = d_init / d_init.sum() * d_true.sum()
i0_init = rng.uniform(100, 5000, N_LENSES)
cf_init = 10.0 ** rng.uniform(-6, -4, N_LENSES)
x0 = np.concatenate([d_init, i0_init, cf_init])

# Run optimization
print(f"\nOptimizing ({N_STARTS} starts)...")
best_loss = np.inf
best_x = None

for start in range(N_STARTS):
    # Randomize initial guess for each start (stay within bounds)
    rng_local = np.random.default_rng(456 + start)
    x0_perturb = x0 * (0.8 + 0.4 * rng_local.random(len(x0)))
    x0_perturb = np.clip(x0_perturb, lower + 1e-6, upper - 1e-6)
    
    t0 = time.time()
    sol = scipy_least_squares(rfn, x0_perturb, bounds=(lower, upper), method='trf', ftol=1e-8, xtol=1e-8, gtol=1e-8, max_nfev=5000, verbose=0)
    elapsed = time.time() - t0
    
    loss = np.sum(np.array(rfn(sol.x))**2)
    if loss < best_loss:
        best_loss = loss
        best_x = sol.x
        x0 = sol.x  # Use best solution as seed for next start
    
    is_best = "← BEST" if loss < best_loss else ""
    print(f"  Start {start+1}: loss={loss:.3e}, time={elapsed:.1f}s {is_best}")

# ================================================================
# REFINEMENT: Additional optimization passes if needed
# ================================================================

if REFINE_RUNS > 0:
    print("Using tighter tolerances...")
    
    for refine_idx in range(REFINE_RUNS):
        t0 = time.time()
        sol = scipy_least_squares(
            rfn, best_x,
            bounds=(lower, upper),
            method='trf',
            ftol=1e-13, xtol=1e-13, gtol=1e-13,
            max_nfev=10000,
            verbose=0
        )
        elapsed = time.time() - t0
        
        loss = np.sum(np.array(rfn(sol.x))**2)
        if loss < best_loss:
            best_loss = loss
            best_x = sol.x
            is_better = "✓ improved"
        else:
            is_better = "  converged"
        
        print(f"  Refine {refine_idx+1}: loss={loss:.3e}, time={elapsed:.1f}s  {is_better}")

# Reconstruct and report
x_true = np.concatenate([d_true, i0_true, cf_true])
x_fit = np.concatenate([best_x[:N_LENSES], np.array([d_true.sum() - np.sum(best_x[:N_LENSES])]), best_x[N_LENSES:]])
errors = 100 * np.abs((x_fit - x_true) / (x_true + 1e-30))

print(f"\n{'='*70}")
print(f"Final Loss: {best_loss:.6e} | Max Error: {np.max(errors):.2f}%")
print(f"{'='*70}")

print(f"\nDistance errors (mm): {errors[:N_LENSES+1]}")
print(f"I0 errors (%):        {errors[N_LENSES+1:2*N_LENSES+1]}")
print(f"Cf errors (%):        {errors[2*N_LENSES+1:]}")
print(f"\nTrue distances (mm):  {d_true * 1e3}")
print(f"Fitted distances (mm): {x_fit[:N_LENSES+1] * 1e3}")
print(f"\nTrue I0:             {i0_true}")
print(f"Fitted I0:            {x_fit[N_LENSES+1:2*N_LENSES+1]}")
print(f"\nTrue Cf:             {cf_true}")
print(f"Fitted Cf:            {x_fit[2*N_LENSES+1:]}")

status = "✓ EXCELLENT" if np.max(errors) < 1 else "△ GOOD" if np.max(errors) < 5 else "✗ POOR"
print(f"\nResult: {status}")


Testing N=2 Lenses with with_B approach...
Final Loss: 8.419394e-30
Distance max error: 0.00%
I0 max error:       0.00%
Cf max error:       0.00%
Overall max error:  0.00%
Status: ✓ EXCELLENT

Testing N=3 Lenses with with_B approach...
Final Loss: 3.369745e-29
Distance max error: 0.00%
I0 max error:       0.00%
Cf max error:       0.00%
Overall max error:  0.00%
Status: ✓ EXCELLENT

Testing N=4 Lenses with with_B approach...
Final Loss: 4.390247e+00
Distance max error: 1752.02%
I0 max error:       173.58%
Cf max error:       309.67%
Overall max error:  1752.02%
Status: ✗ POOR

Testing N=5 Lenses with with_B approach...
Final Loss: 4.025669e-01
Distance max error: 1131.80%
I0 max error:       14.96%
Cf max error:       400.00%
Overall max error:  1131.80%
Status: ✗ POOR


SUMMARY: Fitting Quality vs Number of Lenses (with_B approach)
N    Final Loss       D Error      I0 Error     Cf Error     Status      
----------------------------------------------------------------------
2    8.41

In [ ]:
# ================================================================
# COMPARATIVE TEST: Run all N values and compare
# ================================================================

results_summary = []

for N_LENSES in [2, 3, 4, 5]:
    MEASUREMENT_TYPE = 'clean'
    FITTING_APPROACH = 'with_B'
    N_STARTS = 30  # Reduced for faster testing
    REFINE_RUNS = 10
    
    sys_params = SYSTEMS[N_LENSES]
    d_true = sys_params['D']
    i0_true = sys_params['I0']
    cf_true = sys_params['Cf']
    
    print(f"\n{'='*70}")
    print(f"Testing N={N_LENSES} Lenses with with_B approach...")
    print(f"{'='*70}")
    
    # Generate measurements
    wobble_vals = np.linspace(-0.1, 0.1, 25)
    defocus_vals = np.linspace(0, 0.0, 1)
    meas_clean = generate_measurements(d_true, i0_true, cf_true, K_ROT, wobble_vals, defocus_vals)
    meas = meas_clean
    
    # Create residual function
    scales = (max(np.max(np.abs(meas["A"])), 1e-12), max(np.max(np.abs(meas["B"])), 1e-12), max(np.max(np.abs(meas["psi"])), 1e-12))
    rfn = make_residual_fn_constrained(meas, N_LENSES, scales, K_ROT, total_distance=d_true.sum())
    
    # Optimization bounds
    lower = np.concatenate([np.full(N_LENSES, 1e-3), np.full(N_LENSES, 10.0), np.full(N_LENSES, 1e-6)])
    upper = np.concatenate([np.full(N_LENSES, 0.5), np.full(N_LENSES, 30000.0), np.full(N_LENSES, 1e-4)])
    
    # Initial guess
    rng = np.random.default_rng(123)
    d_init = d_true.sum() * (rng.random(N_LENSES) + 0.1)
    d_init = d_init / d_init.sum() * d_true.sum()
    i0_init = rng.uniform(100, 5000, N_LENSES)
    cf_init = 10.0 ** rng.uniform(-6, -4, N_LENSES)
    x0 = np.concatenate([d_init, i0_init, cf_init])
    
    # Run optimization
    best_loss = np.inf
    best_x = None
    
    for start in range(N_STARTS):
        rng_local = np.random.default_rng(456 + start)
        x0_perturb = x0 * (0.8 + 0.4 * rng_local.random(len(x0)))
        x0_perturb = np.clip(x0_perturb, lower + 1e-6, upper - 1e-6)
        
        sol = scipy_least_squares(rfn, x0_perturb, bounds=(lower, upper), method='trf', ftol=1e-8, xtol=1e-8, gtol=1e-8, max_nfev=5000, verbose=0)
        
        loss = np.sum(np.array(rfn(sol.x))**2)
        if loss < best_loss:
            best_loss = loss
            best_x = sol.x
            x0 = sol.x
    
    # Refinement
    for refine_idx in range(REFINE_RUNS):
        sol = scipy_least_squares(
            rfn, best_x,
            bounds=(lower, upper),
            method='trf',
            ftol=1e-13, xtol=1e-13, gtol=1e-13,
            max_nfev=10000,
            verbose=0
        )
        loss = np.sum(np.array(rfn(sol.x))**2)
        if loss < best_loss:
            best_loss = loss
            best_x = sol.x
    
    # Reconstruct and compute errors
    x_true = np.concatenate([d_true, i0_true, cf_true])
    x_fit = np.concatenate([best_x[:N_LENSES], np.array([d_true.sum() - np.sum(best_x[:N_LENSES])]), best_x[N_LENSES:]])
    errors = 100 * np.abs((x_fit - x_true) / (x_true + 1e-30))
    
    max_error = np.max(errors)
    distance_max_error = np.max(errors[:N_LENSES+1])
    i0_max_error = np.max(errors[N_LENSES+1:2*N_LENSES+1])
    cf_max_error = np.max(errors[2*N_LENSES+1:])
    
    status = "✓ EXCELLENT" if max_error < 1 else "△ GOOD" if max_error < 5 else "✗ POOR"
    
    print(f"Final Loss: {best_loss:.6e}")
    print(f"Distance max error: {distance_max_error:.2f}%")
    print(f"I0 max error:       {i0_max_error:.2f}%")
    print(f"Cf max error:       {cf_max_error:.2f}%")
    print(f"Overall max error:  {max_error:.2f}%")
    print(f"Status: {status}")
    
    results_summary.append({
        'N': N_LENSES,
        'Loss': best_loss,
        'D_error': distance_max_error,
        'I0_error': i0_max_error,
        'Cf_error': cf_max_error,
        'Max_error': max_error,
        'Status': status
    })

print(f"\n\n{'='*70}")
print("SUMMARY: Fitting Quality vs Number of Lenses (with_B approach)")
print(f"{'='*70}")
print(f"{'N':<4} {'Final Loss':<16} {'D Error':<12} {'I0 Error':<12} {'Cf Error':<12} {'Status':<12}")
print("-" * 70)
for r in results_summary:
    print(f"{r['N']:<4} {r['Loss']:<16.3e} {r['D_error']:<12.2f}% {r['I0_error']:<12.2f}% {r['Cf_error']:<12.2f}% {r['Status']:<12}")
